# HaluEval QA: Steering Knowledge Grounding vs. Hallucination

Language models can generate fluent, plausible-sounding answers that are factually
incorrect — a phenomenon known as **hallucination**. The
[HaluEval](https://huggingface.co/datasets/pminervini/HaluEval) QA subset provides
a controlled benchmark for studying this: each sample pairs a background knowledge
passage and question with two alternative answers —

- a **grounded** answer (`right_answer`) extracted faithfully from the passage, and
- a **hallucinated** answer (`hallucinated_answer`) that is plausible but factually wrong.

The model is always prompted with the knowledge passage. We then evaluate two steering targets:

| Target | Goal | Metric |
|---|---|---|
| **Grounded** | Follow the passage; answer faithfully | `grounded_accuracy` |
| **Hallucinated** | Ignore the passage; generate a plausible-but-wrong answer | `hallucination_rate` |

We compare a **baseline** (no steering) against **CAA** (Contrastive Activation Addition)
and **ActAdd** (Activation Addition), both steered in each direction.

Evaluation uses an LLM judge (`HallucinationJudge`) that classifies each response as
"grounded", "hallucinated", or "neither" — avoiding the brittleness of exact-match
scoring for longer answers.

### Runtime Estimate

> **Estimated Time:** 60–120 minutes (200 evaluation samples, 5 pipelines + LLM judge)
> **Device:** NVIDIA A100 / H100 GPU (40 GB+ VRAM)

Reduce `N_EVAL` or `N_STEER` below to speed up experimentation.

## Setup

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()

login(token=os.environ.get("HF_TOKEN", ""))

In [ ]:
from pathlib import Path

import pandas as pd
import transformers
from datasets import load_dataset

from aisteer360.algorithms.state_control.caa.control import CAA
from aisteer360.algorithms.state_control.act_add.control import ActAdd
from aisteer360.algorithms.state_control.common.specs import ContrastivePairs
from aisteer360.evaluation.benchmark import Benchmark
from aisteer360.evaluation.use_cases.halueval.use_case import HaluEvalQA
from aisteer360.evaluation.metrics.custom.halueval.hallucination_judge import HallucinationJudge
from aisteer360.evaluation.utils.data_utils import flatten_profiles

transformers.logging.set_verbosity_error()

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# number of samples for steering vector estimation and evaluation
N_STEER = 500
N_EVAL  = 200

NOTEBOOK_DIR = Path.cwd() / "examples/notebooks/benchmark_halueval"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

## Loading the Data

We load the `qa` subset of [HaluEval](https://huggingface.co/datasets/pminervini/HaluEval),
which contains 10 000 question–passage pairs. We add a string `id` field (required by the
use case) and partition the data:

- **steer split** (first `N_STEER` samples) — used to estimate CAA steering vectors.
- **eval split** (next `N_EVAL` samples) — held out for benchmarking.

In [ ]:
ds = load_dataset("pminervini/HaluEval", "qa", split="data")

records = [
    {
        "id": str(i),
        "question": row["question"],
        "knowledge": row["knowledge"],
        "right_answer": row["right_answer"],
        "hallucinated_answer": row["hallucinated_answer"],
    }
    for i, row in enumerate(ds)
]

steer_records = records[:N_STEER]
eval_records  = records[N_STEER : N_STEER + N_EVAL]

print(f"Steering split : {len(steer_records)} samples")
print(f"Evaluation split: {len(eval_records)} samples")
print()
print("Example record:")
ex = eval_records[0]
print(f"  question           : {ex['question']}")
print(f"  right_answer       : {ex['right_answer']}")
print(f"  hallucinated_answer: {ex['hallucinated_answer']}")
print(f"  knowledge          : {ex['knowledge'][:120]}...")

## Building the Use Case

The `HaluEvalQA` use case always presents the model with the knowledge passage.
The `HallucinationJudge` metric then independently classifies each response as
"grounded" (matching `right_answer`), "hallucinated" (matching `hallucinated_answer`),
or "neither", using an LLM judge — avoiding the brittleness of exact-match scoring
for longer answers.

In [ ]:
use_case = HaluEvalQA(
    evaluation_data=eval_records,
    evaluation_metrics=[HallucinationJudge(model_or_id=MODEL_NAME)],
)

## Preparing Steering Data

### CAA Contrastive Pairs

CAA (Contrastive Activation Addition) estimates a direction in activation space by
contrasting two sets of texts. Critically, **both sets include the same knowledge
passage** — the only difference is which answer is provided:

- **Positives** — `Knowledge: {knowledge} Question: … Answer: {right_answer}`: the model
  sees the passage and the grounded answer.
- **Negatives** — `Knowledge: {knowledge} Question: … Answer: {hallucinated_answer}`: the
  model sees the same passage but the answer is hallucinated.

This pairing isolates the *grounding* direction in activation space while keeping the
passage constant, giving a cleaner contrastive signal.
Applying a **positive multiplier** steers the model toward grounded answers; a
**negative multiplier** steers it toward hallucinated answers.

In [ ]:
positives = [
    f"Knowledge: {r['knowledge']}\nQuestion: {r['question']}\nAnswer: {r['right_answer']}"
    for r in steer_records
]
negatives = [
    f"Knowledge: {r['knowledge']}\nQuestion: {r['question']}\nAnswer: {r['hallucinated_answer']}"
    for r in steer_records
]

caa_pairs = ContrastivePairs(positives=positives, negatives=negatives)

print(f"Contrastive pairs: {len(caa_pairs.positives)}")
print(f"\nPositive example (knowledge + right_answer):\n  {positives[0][:300]}...")
print(f"\nNegative example (knowledge + hallucinated_answer):\n  {negatives[0][:300]}...")

## Defining the Steering Pipelines

We define five pipelines:

| Pipeline | Method | Steering target |
|---|---|---|
| `baseline` | none | — |
| `caa_grounded` | CAA (multiplier > 0) | grounded answers |
| `caa_hallucinated` | CAA (multiplier < 0) | hallucinated answers |
| `act_add_grounded` | ActAdd | grounded answers |
| `act_add_hallucinated` | ActAdd | hallucinated answers |

For CAA we reuse the same contrastive pairs and flip the direction via the sign of
`multiplier`. For ActAdd we supply a prompt pair that captures the same conceptual
contrast, letting the library extract a positional steering vector at inference time.

In [ ]:
CAA_MULTIPLIER     = 20
ACT_ADD_MULTIPLIER = 10

steering_pipelines = {
    "baseline": [],
    "caa_grounded": [
        CAA(
            data=caa_pairs,
            multiplier=CAA_MULTIPLIER,
            token_scope="after_prompt",
        )
    ],
    "caa_hallucinated": [
        CAA(
            data=caa_pairs,
            multiplier=-CAA_MULTIPLIER,
            token_scope="after_prompt",
        )
    ],
    "act_add_grounded": [
        ActAdd(
            positive_prompt="Answer the question faithfully based on the provided knowledge.",
            negative_prompt="Generate a plausible-sounding answer regardless of the knowledge.",
            multiplier=ACT_ADD_MULTIPLIER,
        )
    ],
    "act_add_hallucinated": [
        ActAdd(
            positive_prompt="Generate a plausible-sounding answer regardless of the knowledge.",
            negative_prompt="Answer the question faithfully based on the provided knowledge.",
            multiplier=ACT_ADD_MULTIPLIER,
        )
    ],
}

## Running the Benchmark

The `Benchmark` class loads the model once, applies each pipeline in turn, and saves
incremental checkpoints so the run can be resumed if interrupted.

We use greedy decoding (`do_sample=False`) with a short generation budget
(`max_new_tokens=30`). After generation the `HallucinationJudge` (already loaded
inside the use case) classifies each response.

In [ ]:
benchmark = Benchmark(
    use_case=use_case,
    base_model_name_or_path=MODEL_NAME,
    steering_pipelines=steering_pipelines,
    gen_kwargs={"max_new_tokens": 30, "do_sample": False},
    device_map="cuda",
    save_dir=NOTEBOOK_DIR / "profiles",
)

profiles = benchmark.run()
benchmark.export(profiles, save_dir=NOTEBOOK_DIR / "profiles")

## Results

We extract `grounded_accuracy` and `hallucination_rate` from the profiles and display
a summary table. The two metrics are complementary: a method that strongly steers toward
grounded answers should raise `grounded_accuracy` and lower `hallucination_rate`,
and vice versa.

In [ ]:
df = flatten_profiles(
    profiles,
    metric_accessors={
        "grounded_accuracy":  ("HallucinationJudge", "grounded_accuracy"),
        "hallucination_rate": ("HallucinationJudge", "hallucination_rate"),
    },
)

summary = (
    df.groupby("pipeline")[["grounded_accuracy", "hallucination_rate"]]
    .mean()
    .round(3)
    .reset_index()
)

pipeline_order = list(steering_pipelines.keys())
summary["_order"] = summary["pipeline"].apply(
    lambda p: pipeline_order.index(p) if p in pipeline_order else len(pipeline_order)
)
summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

summary.style.format({
    "grounded_accuracy":  "{:.1%}",
    "hallucination_rate": "{:.1%}",
}).background_gradient(subset=["grounded_accuracy"],  cmap="Blues") \
 .background_gradient(subset=["hallucination_rate"], cmap="Reds")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pipelines   = summary["pipeline"].tolist()
grounded    = summary["grounded_accuracy"].tolist()
hallucinated = summary["hallucination_rate"].tolist()

x = np.arange(len(pipelines))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width / 2, grounded,    width, label="grounded accuracy",  color="steelblue")
bars2 = ax.bar(x + width / 2, hallucinated, width, label="hallucination rate", color="firebrick")

ax.set_xticks(x)
ax.set_xticklabels(pipelines, rotation=20, ha="right")
ax.set_ylabel("Rate (LLM-judge)")
ax.set_ylim(0, 1)
ax.set_title(f"HaluEval QA benchmark — {MODEL_NAME.split('/')[-1]}")
ax.legend()
ax.grid(axis="y", alpha=0.3)

(NOTEBOOK_DIR / "figures").mkdir(parents=True, exist_ok=True)
fig.tight_layout()
fig.savefig(NOTEBOOK_DIR / "figures" / "halueval_qa_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## Inspecting Sample Responses

Browse per-question predictions across pipelines to understand qualitative behavior.

In [ ]:
N_SHOW = 5

rows = []
first_pipeline = next(iter(profiles))
first_gens = profiles[first_pipeline][0]["generations"]

for i in range(min(N_SHOW, len(first_gens))):
    row = {
        "question":           first_gens[i]["question"],
        "right_answer":       first_gens[i]["right_answer"][0],
        "hallucinated_answer": first_gens[i]["hallucinated_answer"][0],
    }
    for pipeline, runs in profiles.items():
        row[pipeline] = runs[0]["generations"][i]["response"]
    rows.append(row)

pd.DataFrame(rows).T

## Takeaways

- The **baseline** typically has moderate `grounded_accuracy` — the model reads the
  passage but may still produce hallucinated answers due to prior training biases.
- **CAA with a positive multiplier** pushes activations toward the grounding direction,
  increasing `grounded_accuracy` at the cost of `hallucination_rate`.
- **CAA with a negative multiplier** (hallucination steering) has the opposite effect.
- **ActAdd** achieves a similar directional effect via a single prompt-pair vector,
  making it simpler to configure but potentially less precise than data-driven CAA.
- Responses classified as "neither" by the judge are counted in neither metric;
  a high "neither" rate may indicate fluency degradation from excessive steering strength.